Fully Connected Segmentation + SegGBC/AUV

This experiment uses a pixel-wise multilayer perceptron with no convolutional
layers. A second trainable pixel head consumes the frozen MLP features plus a
per-pixel AUV map derived from SegGBC balls. Results compare the baseline MLP,
SegGBC majority refinement, and the trained AUV-guided head. Granular-ball
processing is limited to configurable subsets because it is computationally costly.


In [1]:
import hashlib
import math
import random
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.spatial import cKDTree
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data_dir = Path('data/3/BraTS2020_training_data/content/data')
if not data_dir.is_dir():
    raise FileNotFoundError(f'Dataset directory not found: {data_dir.resolve()}')


class BraTSSliceDataset(Dataset):
    def __init__(self, files):
        self.files = list(files)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        path = self.files[index]
        with h5py.File(path, 'r') as h5_file:
            image = np.asarray(h5_file['image'][...], dtype=np.float32)
            mask = np.asarray(h5_file['mask'][...], dtype=np.float32)
        scale = np.max(np.abs(image), axis=(0, 1), keepdims=True)
        scale[scale < 1e-6] = 1.0
        image = np.clip(image / scale, -1.0, 1.0)
        if mask.ndim == 3:
            mask = np.argmax(mask, axis=-1)
        return (
            torch.from_numpy(image).permute(2, 0, 1),
            torch.from_numpy(mask.astype(np.int64)),
            str(path),
        )


h5_files = list(data_dir.glob('*.h5'))
if not h5_files:
    raise FileNotFoundError(f'No .h5 files found under {data_dir.resolve()}')
random.shuffle(h5_files)
train_end = int(0.70 * len(h5_files))
val_end = train_end + int(0.15 * len(h5_files))
train_files = h5_files[:train_end]
val_files = h5_files[train_end:val_end]
test_files = h5_files[val_end:]

batch_size = 4
train_loader = DataLoader(BraTSSliceDataset(train_files), batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(BraTSSliceDataset(val_files), batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(BraTSSliceDataset(test_files), batch_size=batch_size, shuffle=False, num_workers=2)
print(f'Device: {device}')
print(f'Train: {len(train_files)} | Validation: {len(val_files)} | Test: {len(test_files)}')


Device: cuda
Train: 40036 | Validation: 8579 | Test: 8580


In [4]:
class FullyConnectedSegmenter(nn.Module):
    """Shared per-pixel MLP; all learned layers are Linear (no convolution)."""
    def __init__(self, in_channels=4, hidden_channels=32, num_classes=4):
        super().__init__()
        self.hidden_channels = hidden_channels
        self.encoder = nn.Sequential(
            nn.Linear(in_channels + 2, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, hidden_channels),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Linear(hidden_channels, num_classes)

    def forward(self, image):
        batch, _, height, width = image.shape
        y, x = torch.meshgrid(
            torch.linspace(-1, 1, height, device=image.device, dtype=image.dtype),
            torch.linspace(-1, 1, width, device=image.device, dtype=image.dtype),
            indexing='ij',
        )
        coords = torch.stack((x, y), dim=-1).unsqueeze(0).expand(batch, -1, -1, -1)
        pixels = torch.cat((image.permute(0, 2, 3, 1), coords), dim=-1)
        hidden = self.encoder(pixels.reshape(-1, pixels.shape[-1]))
        logits = self.classifier(hidden)
        hidden_map = hidden.reshape(batch, height, width, self.hidden_channels).permute(0, 3, 1, 2).contiguous()
        logits = logits.reshape(batch, height, width, -1).permute(0, 3, 1, 2).contiguous()
        return logits, hidden_map


num_classes = 4
model = FullyConnectedSegmenter(in_channels=4, hidden_channels=32, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 10

checkpoint_dir = Path('checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)
latest_checkpoint = checkpoint_dir / 'fcn_segmentation_latest.pth'
final_model_path = Path('fcn_segmentation_final.pth')
start_epoch = 1
if latest_checkpoint.exists():
    checkpoint = torch.load(latest_checkpoint, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    print(f'Resuming from epoch {start_epoch}')
else:
    print('No FCN checkpoint found; training from scratch.')

for epoch in range(start_epoch, num_epochs + 1):
    model.train()
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{num_epochs}', unit='batch')
    for batch_idx, (images, masks, _) in enumerate(progress):
        images = images.to(device)
        masks = masks.to(device=device, dtype=torch.long)
        optimizer.zero_grad(set_to_none=True)
        logits, _ = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        progress.set_postfix(loss=f'{running_loss / (batch_idx + 1):.4f}')
    model.eval()
    validation_loss = 0.0
    with torch.inference_mode():
        for images, masks, _ in val_loader:
            images = images.to(device)
            masks = masks.to(device=device, dtype=torch.long)
            logits, _ = model(images)
            validation_loss += criterion(logits, masks).item()
    validation_loss /= max(len(val_loader), 1)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': running_loss / max(len(train_loader), 1),
        'validation_loss': validation_loss,
    }
    temp_path = latest_checkpoint.with_suffix('.tmp')
    torch.save(checkpoint, temp_path)
    temp_path.replace(latest_checkpoint)
    print(f'Validation loss: {validation_loss:.4f} | Saved checkpoint: {latest_checkpoint}')

torch.save(model.state_dict(), final_model_path)
print(f'Saved final model weights: {final_model_path}')


No FCN checkpoint found; training from scratch.


Epoch 1/10:   0%|          | 0/10009 [00:00<?, ?batch/s]

Validation loss: 0.0270 | Saved checkpoint: checkpoints/fcn_segmentation_latest.pth


Epoch 2/10:   0%|          | 0/10009 [00:00<?, ?batch/s]

Validation loss: 0.0268 | Saved checkpoint: checkpoints/fcn_segmentation_latest.pth


Epoch 3/10:   0%|          | 0/10009 [00:00<?, ?batch/s]

... (13 outputs left out)

Epoch 9/10:   0%|          | 0/10009 [00:00<?, ?batch/s]

Validation loss: 0.0261 | Saved checkpoint: checkpoints/fcn_segmentation_latest.pth


Epoch 10/10:   0%|          | 0/10009 [00:00<?, ?batch/s]

Validation loss: 0.0259 | Saved checkpoint: checkpoints/fcn_segmentation_latest.pth
Saved final model weights: fcn_segmentation_final.pth


In [5]:
import hashlib

class GranularBall:
    def __init__(self, points):
        self.points = np.asarray(points, dtype=np.float64)
        self.dim = self.points.shape[1]
        self.center = self.points.mean(axis=0)
        self.pi = np.zeros(len(self.points), dtype=np.float64)
        self.radius_max = self.radius_avg = self.radius_sta = 0.0
        self.scm = 1.0

    def update(self, alpha=0.7):
        diffs = self.points - self.center
        dist_sq = np.sum(diffs * diffs, axis=1)
        max_dist = float(np.max(dist_sq)) if len(dist_sq) > 1 else 1e-5
        sigma_m = max(alpha * math.sqrt(max_dist + 1e-8), 1e-5)
        sigma_n = max((1.0 - alpha) * math.sqrt(max_dist + 1e-8), 1e-5)
        mu = np.exp(-dist_sq / (sigma_m * sigma_m))
        nu = 1.0 - np.exp(-dist_sq / (sigma_n * sigma_n))
        self.pi = np.clip(1.0 - mu - nu, 0.0, 1.0)
        self.center = np.mean(self.points + self.pi[:, None], axis=0)
        diffs = self.points - self.center
        dist_pi = np.sum(diffs * diffs, axis=1) + self.pi
        self.radius_max = max(float(np.max(dist_pi)), 1e-8)
        self.radius_avg = max(float(np.mean(dist_pi)), 1e-8)
        self.radius_sta = max(self.radius_avg * 0.5, 1e-8)
        return dist_pi


class SegGBC:
    """Granular-ball splitting/merging over a feature map with any channel count."""
    def __init__(self, beta=0.8, epsilon=0.75, alpha=0.7, max_iter=30, max_balls=64):
        self.beta, self.epsilon, self.alpha = beta, epsilon, alpha
        self.max_iter, self.max_balls = max_iter, max_balls
        self.balls = []

    def fit(self, feature_map):
        if feature_map.ndim != 3:
            raise ValueError(f'Expected [height, width, channels], got {feature_map.shape}')
        self.height, self.width, self.feature_channels = feature_map.shape
        self.spatial_weight = 0.1
        scales = np.max(np.abs(feature_map), axis=(0, 1), keepdims=True)
        scales[scales < 1e-8] = 1.0
        features = feature_map / scales
        yy, xx = np.meshgrid(np.arange(self.height), np.arange(self.width), indexing='ij')
        points = np.column_stack((
            features.reshape(-1, self.feature_channels),
            xx.reshape(-1) / self.width * self.spatial_weight,
            yy.reshape(-1) / self.height * self.spatial_weight,
        ))
        self.balls = [GranularBall(points)]
        for _ in range(self.max_iter):
            new_balls = []
            split_occurred = False
            for ball_index, ball in enumerate(self.balls):
                dist_pi = ball.update(self.alpha)
                dim = ball.dim
                def log_density(count, radius):
                    return math.log(max(int(count), 1)) - dim * math.log(max(float(radius), 1e-8))
                rho_max = log_density(len(ball.points), ball.radius_max)
                rho_avg = log_density(np.count_nonzero(dist_pi <= ball.radius_avg), ball.radius_avg)
                rho_sta = log_density(np.count_nonzero(dist_pi <= ball.radius_sta), ball.radius_sta)
                log_scm = min(rho_avg, rho_max) + min(rho_sta, rho_max) - max(rho_avg, rho_max) - max(rho_sta, rho_max)
                ball.scm = math.exp(max(-745.0, min(0.0, log_scm)))
                projected_count = len(new_balls) + 2 + len(self.balls) - ball_index - 1
                if ball.scm < self.beta and len(ball.points) > 20 and projected_count <= self.max_balls:
                    axis = int(np.argmax(np.var(ball.points, axis=0)))
                    order = np.argsort(ball.points[:, axis])
                    midpoint = len(order) // 2
                    new_balls.extend((
                        GranularBall(ball.points[order[:midpoint]]),
                        GranularBall(ball.points[order[midpoint:]]),
                    ))
                    split_occurred = True
                else:
                    new_balls.append(ball)
            self.balls = new_balls
            if not split_occurred:
                break
        self._merge_clusters()
        return self._labels_from_points()

    def _merge_clusters(self):
        if len(self.balls) < 2:
            return
        centers = np.asarray([ball.center for ball in self.balls])
        mean_radius = float(np.mean([ball.radius_max for ball in self.balls]))
        pairs = cKDTree(centers).query_pairs(r=max(mean_radius * 3.0, 1.0))
        removed = set()
        for i, j in sorted(pairs):
            if i in removed or j in removed:
                continue
            distance = np.linalg.norm(self.balls[i].center - self.balls[j].center)
            if 1.0 / (1.0 + distance) > self.epsilon:
                self.balls[i].points = np.vstack((self.balls[i].points, self.balls[j].points))
                self.balls[i].update(self.alpha)
                removed.add(j)
        self.balls = [ball for index, ball in enumerate(self.balls) if index not in removed]

    def _labels_from_points(self):
        labels = np.zeros((self.height, self.width), dtype=np.int32)
        for label, ball in enumerate(self.balls):
            x = np.rint(ball.points[:, self.feature_channels] * self.width / self.spatial_weight).astype(np.int64)
            y = np.rint(ball.points[:, self.feature_channels + 1] * self.height / self.spatial_weight).astype(np.int64)
            valid = (x >= 0) & (x < self.width) & (y >= 0) & (y < self.height)
            labels[y[valid], x[valid]] = label
        return labels


def auv_by_ball(gbc):
    """Compute each ball's AUV from its MLP feature dimensions only.

    SegGBC appends two spatial coordinates to every point; these are excluded
    here so AUV describes hidden-feature variation, not coordinate variation.
    """
    scores = []
    for ball in gbc.balls:
        features = ball.points[:, :gbc.feature_channels]
        if len(features) < 2:
            scores.append(0.0)
            continue
        centered = features - features.mean(axis=0, keepdims=True)
        energy = np.clip(np.linalg.eigvalsh(centered.T @ centered), 0.0, None)
        energy_sum = float(energy.sum())
        positive = energy[energy > 1e-12]
        if energy_sum <= 1e-12 or len(positive) <= 1:
            scores.append(0.0)
        else:
            probabilities = positive / energy_sum
            scores.append(float(-np.sum(probabilities * np.log(probabilities)) / np.log(len(positive))))
    return scores


def auv_map_from_mlp_features(mlp_feature_map):
    """Cluster an MLP feature map and broadcast each ball's AUV to its pixels."""
    gbc = SegGBC(beta=0.8, epsilon=0.75, alpha=0.7)
    cluster_labels = gbc.fit(mlp_feature_map)
    auv_scores = auv_by_ball(gbc)
    auv_map = np.zeros(cluster_labels.shape, dtype=np.float32)
    for cluster_id, score in enumerate(auv_scores):
        auv_map[cluster_labels == cluster_id] = score
    return auv_map, gbc, cluster_labels, auv_scores


def new_totals():
    return tuple(np.zeros(num_classes, dtype=np.float64) for _ in range(4))


def add_metrics(totals, target, prediction):
    intersection, union, target_area, pred_area = totals
    for class_id in range(num_classes):
        pred_class = prediction == class_id
        target_class = target == class_id
        intersection[class_id] += np.logical_and(pred_class, target_class).sum()
        union[class_id] += np.logical_or(pred_class, target_class).sum()
        target_area[class_id] += target_class.sum()
        pred_area[class_id] += pred_class.sum()


def summarize_metrics(totals):
    intersection, union, target_area, pred_area = totals
    dice_denominator = target_area + pred_area
    dice = np.divide(2 * intersection, dice_denominator, out=np.ones_like(intersection), where=dice_denominator > 0)
    iou = np.divide(intersection, union, out=np.ones_like(intersection), where=union > 0)
    return {
        'class_dice': dice,
        'class_iou': iou,
        'mean_tumor_dice': float(dice[1:].mean()),
        'mean_tumor_iou': float(iou[1:].mean()),
    }


class AUVGuidedSegmentationHead(nn.Module):
    """Per-pixel MLP head that predicts classes from hidden features and AUV."""
    def __init__(self, feature_channels, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(feature_channels + 1, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes),
        )

    def forward(self, feature_map, auv_map):
        batch, channels, height, width = feature_map.shape
        pixels = feature_map.permute(0, 2, 3, 1)
        pixels = torch.cat((pixels, auv_map.unsqueeze(-1)), dim=-1)
        logits = self.layers(pixels.reshape(-1, channels + 1))
        return logits.reshape(batch, height, width, -1).permute(0, 3, 1, 2).contiguous()


def soft_dice_loss(logits, targets, smooth=1e-6):
    """Mean multiclass soft Dice loss over classes and batch pixels."""
    probabilities = torch.softmax(logits, dim=1)
    one_hot_targets = F.one_hot(targets, num_classes=logits.shape[1]).permute(0, 3, 1, 2).to(probabilities.dtype)
    intersection = (probabilities * one_hot_targets).sum(dim=(0, 2, 3))
    denominator = probabilities.sum(dim=(0, 2, 3)) + one_hot_targets.sum(dim=(0, 2, 3))
    dice_per_class = (2.0 * intersection + smooth) / (denominator + smooth)
    return 1.0 - dice_per_class.mean()


if not final_model_path.exists():
    raise FileNotFoundError(f'{final_model_path} not found. Run the training cell first.')
model.load_state_dict(torch.load(final_model_path, map_location=device))
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)

# GBC/AUV maps are expensive to compute. Cache them for a configurable training
# subset; set either limit to None to use the full corresponding split.
AUV_HEAD_TRAIN_LIMIT = 500
AUV_HEAD_VAL_LIMIT = 100
auv_head_train_files = train_files[:AUV_HEAD_TRAIN_LIMIT] if AUV_HEAD_TRAIN_LIMIT is not None else train_files
auv_head_val_files = val_files[:AUV_HEAD_VAL_LIMIT] if AUV_HEAD_VAL_LIMIT is not None else val_files
model_stat = final_model_path.stat()
auv_cache_dir = Path('auv_feature_cache') / f'{model_stat.st_size}_{model_stat.st_mtime_ns}'
auv_cache_dir.mkdir(parents=True, exist_ok=True)

def auv_cache_path(file_path):
    cache_name = hashlib.sha1(str(Path(file_path).resolve()).encode('utf-8')).hexdigest()
    return auv_cache_dir / f'{cache_name}.npy'

def ensure_auv_cache(files):
    source_dataset = BraTSSliceDataset(files)
    for index, file_path in enumerate(tqdm(files, desc='Preparing MLP-feature AUV maps', unit='image')):
        cache_path = auv_cache_path(file_path)
        if cache_path.exists():
            continue
        image, _, _ = source_dataset[index]
        with torch.inference_mode():
            _, latent = model(image.unsqueeze(0).to(device))
        feature_map = latent[0].permute(1, 2, 0).cpu().numpy()
        auv_map, _, _, _ = auv_map_from_mlp_features(feature_map)
        temp_path = cache_path.with_suffix('.tmp')
        with temp_path.open('wb') as cache_file:
            np.save(cache_file, auv_map.astype(np.float16))
        temp_path.replace(cache_path)

class AUVFeatureDataset(Dataset):
    def __init__(self, files):
        self.base_dataset = BraTSSliceDataset(files)
        self.files = list(files)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        image, mask, file_path = self.base_dataset[index]
        auv_map = np.load(auv_cache_path(file_path)).astype(np.float32)
        return image, mask, torch.from_numpy(auv_map), file_path

ensure_auv_cache(auv_head_train_files)
ensure_auv_cache(auv_head_val_files)
auv_train_loader = DataLoader(AUVFeatureDataset(auv_head_train_files), batch_size=4, shuffle=True, num_workers=2)
auv_val_loader = DataLoader(AUVFeatureDataset(auv_head_val_files), batch_size=4, shuffle=False, num_workers=2)
auv_head = AUVGuidedSegmentationHead(feature_channels=model.hidden_channels, num_classes=num_classes).to(device)
auv_head_optimizer = torch.optim.Adam(auv_head.parameters(), lr=1e-3)
auv_head_epochs = 10
auv_head_dice_weight = 1.0
best_auv_head_val_loss = float('inf')
auv_head_path = Path('auv_guided_segmentation_head.pth')

for epoch in range(1, auv_head_epochs + 1):
    auv_head.train()
    train_loss = 0.0
    for batch_idx, (images, masks, auv_maps, _) in enumerate(tqdm(auv_train_loader, desc=f'AUV head epoch {epoch}/{auv_head_epochs}', unit='batch')):
        images = images.to(device)
        masks = masks.to(device=device, dtype=torch.long)
        auv_maps = auv_maps.to(device=device, dtype=torch.float32)
        with torch.no_grad():
            _, latent = model(images)
        auv_head_optimizer.zero_grad(set_to_none=True)
        head_logits = auv_head(latent, auv_maps)
        loss = criterion(head_logits, masks) + auv_head_dice_weight * soft_dice_loss(head_logits, masks)
        loss.backward()
        auv_head_optimizer.step()
        train_loss += loss.item()

    auv_head.eval()
    val_loss = 0.0
    with torch.inference_mode():
        for images, masks, auv_maps, _ in auv_val_loader:
            images = images.to(device)
            masks = masks.to(device=device, dtype=torch.long)
            auv_maps = auv_maps.to(device=device, dtype=torch.float32)
            _, latent = model(images)
            head_logits = auv_head(latent, auv_maps)
            val_loss += (criterion(head_logits, masks) + auv_head_dice_weight * soft_dice_loss(head_logits, masks)).item()
    val_loss /= max(len(auv_val_loader), 1)
    print(f'AUV head epoch {epoch}: train loss={train_loss / max(len(auv_train_loader), 1):.4f}, val loss={val_loss:.4f}')
    if val_loss < best_auv_head_val_loss:
        best_auv_head_val_loss = val_loss
        torch.save(auv_head.state_dict(), auv_head_path)

auv_head.load_state_dict(torch.load(auv_head_path, map_location=device))
auv_head.eval()
print(f'Loaded best validation AUV-guided head: {auv_head_path}')

baseline_totals = new_totals()
subset_baseline_totals = new_totals()
gbc_totals = new_totals()
auv_head_totals = new_totals()
gbc_max_test_files = min(100, len(test_files))
auv_image_means = []
cluster_counts = []
processed = 0

with torch.inference_mode():
    for images, masks, _ in tqdm(test_loader, desc='Testing FCN + SegGBC/AUV', unit='batch'):
        images = images.to(device)
        logits, latent = model(images)
        predictions = logits.argmax(dim=1).cpu().numpy()
        masks_np = masks.numpy()
        for item in range(len(images)):
            target, prediction = masks_np[item], predictions[item]
            add_metrics(baseline_totals, target, prediction)
            if processed < gbc_max_test_files:
                add_metrics(subset_baseline_totals, target, prediction)
                mlp_feature_map = latent[item].permute(1, 2, 0).cpu().numpy()
                auv_map, gbc, cluster_labels, scores = auv_map_from_mlp_features(mlp_feature_map)
                auv_tensor = torch.from_numpy(auv_map).unsqueeze(0).to(device=device, dtype=torch.float32)
                head_logits = auv_head(latent[item:item + 1], auv_tensor)
                auv_head_prediction = head_logits.argmax(dim=1).cpu().numpy()[0]
                add_metrics(auv_head_totals, target, auv_head_prediction)
                refined = np.zeros_like(prediction)
                for cluster_id in range(len(gbc.balls)):
                    region = cluster_labels == cluster_id
                    if np.any(region):
                        counts = np.bincount(prediction[region], minlength=num_classes)
                        refined[region] = int(counts.argmax())
                add_metrics(gbc_totals, target, refined)
                auv_image_means.append(float(np.mean(scores)) if scores else 0.0)
                cluster_counts.append(len(gbc.balls))
            processed += 1

baseline_results = summarize_metrics(baseline_totals)
paired_baseline_results = summarize_metrics(subset_baseline_totals)
gbc_results = summarize_metrics(gbc_totals)
auv_head_results = summarize_metrics(auv_head_totals)
print('\nFCN baseline (full test set)')
for class_id in range(num_classes):
    print(f'Class {class_id}: Dice={baseline_results["class_dice"][class_id]:.4f}, IoU={baseline_results["class_iou"][class_id]:.4f}')
print(f'Mean tumor Dice={baseline_results["mean_tumor_dice"]:.4f}, IoU={baseline_results["mean_tumor_iou"]:.4f}')
print(f'\nPaired comparison on {gbc_max_test_files} test files')
print(f'FCN baseline: tumor Dice={paired_baseline_results["mean_tumor_dice"]:.4f}, IoU={paired_baseline_results["mean_tumor_iou"]:.4f}')
print(f'FCN + SegGBC: tumor Dice={gbc_results["mean_tumor_dice"]:.4f}, IoU={gbc_results["mean_tumor_iou"]:.4f}')
print(f'FCN + SegGBC + trained AUV head: tumor Dice={auv_head_results["mean_tumor_dice"]:.4f}, IoU={auv_head_results["mean_tumor_iou"]:.4f}')
for class_id in range(num_classes):
    print(
        f'Class {class_id} paired: '
        f'FCN Dice/IoU={paired_baseline_results["class_dice"][class_id]:.4f}/'
        f'{paired_baseline_results["class_iou"][class_id]:.4f} | '
        f'FCN+SegGBC Dice/IoU={gbc_results["class_dice"][class_id]:.4f}/'
        f'{gbc_results["class_iou"][class_id]:.4f} | '
        f'FCN+trained AUV head Dice/IoU={auv_head_results["class_dice"][class_id]:.4f}/'
        f'{auv_head_results["class_iou"][class_id]:.4f}'
    )
print(f'Mean AUV across balls/images: {np.mean(auv_image_means):.4f}')
print(f'Mean granular balls per image: {np.mean(cluster_counts):.2f}')

comparison_results = {
    'fcn_full_test': baseline_results,
    'fcn_paired_subset': paired_baseline_results,
    'fcn_plus_seggbc_subset': gbc_results,
    'fcn_plus_trained_auv_head_subset': auv_head_results,
    'auv_mean_per_image': float(np.mean(auv_image_means)) if auv_image_means else 0.0,
    'mean_granular_balls': float(np.mean(cluster_counts)) if cluster_counts else 0.0,
    'seggbc_test_files': gbc_max_test_files,
}


Preparing MLP-feature AUV maps:   0%|          | 0/500 [00:00<?, ?image/s]

Preparing MLP-feature AUV maps:   0%|          | 0/100 [00:00<?, ?image/s]

AUV head epoch 1/10:   0%|          | 0/125 [00:00<?, ?batch/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7aed3b642020>
Traceback (most recent call last):
  File "/home/aman_kumar/anaconda3/envs/ml/lib/python3.13/site-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/home/aman_kumar/anaconda3/envs/ml/lib/python3.13/site-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/home/aman_kumar/anaconda3/envs/ml/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process


AUV head epoch 1: train loss=0.9513, val loss=0.7413


AUV head epoch 2/10:   0%|          | 0/125 [00:00<?, ?batch/s]

... (24 outputs left out)

AUV head epoch 10: train loss=0.6508, val loss=0.6932
Loaded best validation AUV-guided head: auv_guided_segmentation_head.pth


Testing FCN + SegGBC/AUV:   0%|          | 0/2145 [00:00<?, ?batch/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7aed3b642020>
Traceback (most recent call last):
  File "/home/aman_kumar/anaconda3/envs/ml/lib/python3.13/site-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/home/aman_kumar/anaconda3/envs/ml/lib/python3.13/site-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/home/aman_kumar/anaconda3/envs/ml/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7aed3b642020>
Traceback (most recent call last):
  File "/home/aman_kumar/anaconda3/envs/ml/lib/python3.13/site-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/home/aman_kumar/anaconda3/envs/ml/lib/py


FCN baseline (full test set)
Class 0: Dice=0.9961, IoU=0.9922
Class 1: Dice=0.3076, IoU=0.1818
Class 2: Dice=0.2498, IoU=0.1427
Class 3: Dice=1.0000, IoU=1.0000
Mean tumor Dice=0.5191, IoU=0.4415

Paired comparison on 100 test files
FCN baseline: tumor Dice=0.4652, IoU=0.4072
FCN + SegGBC: tumor Dice=0.4260, IoU=0.3837
FCN + SegGBC + trained AUV head: tumor Dice=0.5438, IoU=0.4588
Class 0 paired: FCN Dice/IoU=0.9955/0.9909 | FCN+SegGBC Dice/IoU=0.9954/0.9909 | FCN+trained AUV head Dice/IoU=0.9944/0.9889
Class 1 paired: FCN Dice/IoU=0.2524/0.1444 | FCN+SegGBC Dice/IoU=0.1926/0.1066 | FCN+trained AUV head Dice/IoU=0.3568/0.2171
Class 2 paired: FCN Dice/IoU=0.1434/0.0772 | FCN+SegGBC Dice/IoU=0.0852/0.0445 | FCN+trained AUV head Dice/IoU=0.2747/0.1592
Class 3 paired: FCN Dice/IoU=1.0000/1.0000 | FCN+SegGBC Dice/IoU=1.0000/1.0000 | FCN+trained AUV head Dice/IoU=1.0000/1.0000
Mean AUV across balls/images: 0.3522
Mean granular balls per image: 20.73
